# Ask Question — 知识问答记录

---

## 问题一：欠拟合和过拟合

**提问时间**：2026-06-16

### 欠拟合 (Underfitting)

| 方面 | 内容 |
|------|------|
| **表现** | 训练集上表现差，测试集上也差 |
| **本质** | 模型太简单，没学到数据规律；高偏差 (High Bias) |

**产生原因**：
1. 模型复杂度过低（如用线性模型拟合非线性数据）
2. 特征不足，信息量不够
3. 训练轮次不够
4. 正则化过强（L1/L2 λ 太大、Dropout 过高）
5. 数据预处理问题（未归一化等）

**解决方法**：
1. 增加模型复杂度（换更强模型、加深加宽网络）
2. 优化特征工程（增加特征、多项式特征、交叉特征）
3. 调整训练策略（增加 epoch、调小学习率）
4. 减轻正则化（减小 λ、降低 Dropout rate）

### 过拟合 (Overfitting)

| 方面 | 内容 |
|------|------|
| **表现** | 训练集上表现极好，测试集上表现差 |
| **本质** | 模型太复杂，把噪声也学进去了；高方差 (High Variance) |

**产生原因**：
1. 模型复杂度过高（参数 >> 样本数）
2. 训练数据太少或噪声大
3. 特征过多引发维数灾难
4. 训练过度，没有早停
5. 缺乏正则化手段

**解决方法**：
1. **增加训练数据**（收集更多数据、数据增强、合成数据）
2. **降低模型复杂度**（减层数/神经元、特征选择、剪枝）
3. **正则化**：L1（稀疏解）、L2（权重衰减）、Elastic Net
4. **Dropout**：训练时随机丢弃神经元（常用 rate: 0.2-0.5）
5. **早停 (Early Stopping)**：监控验证集 loss，不再下降时停止
6. **集成学习**：Bagging (Random Forest)、Boosting (XGBoost/LightGBM)
7. **Batch Normalization**：有一定正则化效果
8. **交叉验证**：k-fold 评估泛化能力

### 偏差-方差权衡 (Bias-Variance Tradeoff)

- **偏差 (Bias)**：预测值与真实值的系统性偏离 → 欠拟合 → 高偏差
- **方差 (Variance)**：不同训练集上预测的波动程度 → 过拟合 → 高方差
- **总误差** = Bias² + Variance + 不可约误差
- 最佳模型复杂度在总误差最低点，即偏差和方差平衡处

### 实战诊断

| 训练 Loss | 验证 Loss | 诊断 |
|-----------|-----------|------|
| 高 | 高 | 欠拟合 |
| 低 | 高 | 过拟合 |
| 低 | 低 | ✅ 泛化良好 |

### 学习曲线诊断代码

```python
from sklearn.model_selection import learning_curve
from sklearn.linear_model import Ridge
import numpy as np
import matplotlib.pyplot as plt

train_sizes, train_scores, val_scores = learning_curve(
    estimator=model,
    X=X, y=y,
    train_sizes=np.linspace(0.1, 1.0, 10),
    cv=5,
    scoring='neg_mean_squared_error'
)

train_mean = -train_scores.mean(axis=1)
val_mean = -val_scores.mean(axis=1)

plt.plot(train_sizes, train_mean, 'o-', label='Training error')
plt.plot(train_sizes, val_mean, 'o-', label='Validation error')
plt.xlabel('Training set size')
plt.ylabel('MSE')
plt.legend()
plt.show()
```

---

## 问题二：早停 (Early Stopping) 是什么，为什么能起作用

**提问时间**：2026-06-16

### 定义

早停是一种**训练时正则化技术**：每轮训练后评估验证集表现，当验证误差不再下降（甚至上升）时立即停止训练，取验证 Loss 最低处的模型参数。

```
        Loss
         ↑
         |  Train Loss（持续下降）
         | \
         |  \        ___
         |   \      /   ← 验证 Loss 开始上升
         |    \    /
         |     \  /
         |      \/  ← ★ 早停点（验证 Loss 最低处）
         |
         +——————————→ Epochs
```

### 为什么能起作用

**直观理解**（学生学习类比）：
- 训练初期：模型学数据的"大规律" → 训练误差↓，验证误差↓
- 训练后期：模型开始学训练集特有的噪声和偶然模式 → 训练误差↓，验证误差↑
- 早停在"学完大规律、开始学噪声之前"叫停

**数学直觉**：
- 网络权重从接近 0 逐步增长
- 迭代中期权重适中 → 泛化好
- 迭代过多权重过大 → 对微小波动过度敏感 → 过拟合
- 早停限制了参数的有效搜索空间，不让权重变得过大

**深层原理**：
- 早停在数学上等价于隐式的 L2 正则化
- Bishop (1995) 证明：在线性模型 + 梯度下降 + MSE 条件下，早停等价于特定 λ 的 L2 正则化
- 它通过限制迭代步数来约束参数的有效搜索范围，降低模型的"有效复杂度"

In [ ]:
# PyTorch 早停实现
import torch
import numpy as np

class EarlyStopping:
    def __init__(self, patience=7, min_delta=0, path='checkpoint.pt'):
        """
        patience: 允许多少个 epoch 不改善后停止
        min_delta: 最小改善阈值，小于该值视为"没有改善"
        path: 模型保存路径
        """
        self.patience = patience
        self.min_delta = min_delta
        self.path = path
        self.counter = 0
        self.best_score = None
        self.early_stop = False
    
    def __call__(self, val_loss, model):
        score = -val_loss  # 取负数，因为我们要最大化 score
        
        if self.best_score is None:
            self.best_score = score
            self.save_checkpoint(model)
        elif score < self.best_score + self.min_delta:
            # 没有显著改善
            self.counter += 1
            print(f'EarlyStopping counter: {self.counter} out of {self.patience}')
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            # 有改善，重置计数器
            self.best_score = score
            self.save_checkpoint(model)
            self.counter = 0
    
    def save_checkpoint(self, model):
        torch.save(model.state_dict(), self.path)

In [ ]:
# Keras/TensorFlow 早停用法
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor='val_loss',        # 监控的指标
    patience=10,               # 容忍的 epoch 数
    restore_best_weights=True, # 自动恢复最佳权重
    min_delta=0.001            # 最小变化阈值
)

# model.fit(
#     X_train, y_train,
#     validation_data=(X_val, y_val),
#     epochs=500,
#     callbacks=[early_stop]
# )

### 关键参数

| 参数 | 含义 | 建议值 |
|------|------|--------|
| patience | 容忍多少个 epoch 不改善 | 5~20 |
| min_delta | 改善的最小阈值 | 0.001 或 1e-4 |

### 优点与局限

**优点**：零额外开销、自动选择 epoch 数、效率高、可与其他正则化组合

**局限**：需要验证集、SGD 震荡可能误触发、对严重过拟合效果有限

### 一句话总结

> 早停 = 在模型开始"背答案"之前叫停，确保它学到的是"规律"而不是"噪声"。通过限制优化的迭代步数来约束参数的有效搜索范围，防止过拟合。